# Indian-language audio (m4a) to text with Sarvam AI

Outputs the **detected language**, the **transcript in the original language**, and an **English translation**.

**Setup (once):**
1. Get an API key at https://dashboard.sarvam.ai
2. In Colab, click the key icon (Secrets) in the left sidebar, add a secret named `SARVAM_API_KEY`, and enable notebook access.

The REST endpoint only handles short clips (~30 s), so this notebook splits long audio into chunks and joins the results.

In [ ]:
!pip -q install pydub requests
# ffmpeg is preinstalled on Colab (needed by pydub for m4a)

In [ ]:
import io, requests
from google.colab import userdata, files
from pydub import AudioSegment

API_KEY = userdata.get('SARVAM_API_KEY')
URL = 'https://api.sarvam.ai/speech-to-text'

# 'unknown' = auto-detect language. Or set e.g. 'hi-IN', 'ta-IN', 'te-IN', 'kn-IN', 'bn-IN', 'mr-IN'.
LANGUAGE = 'unknown'
MODEL = 'saaras:v3'
CHUNK_SECONDS = 25

In [ ]:
uploaded = files.upload()  # pick your .m4a file
path = next(iter(uploaded))
audio = AudioSegment.from_file(path)
print(f'{path}: {len(audio)/1000:.1f} s')

In [ ]:
def call_sarvam(seg, mode):
    buf = io.BytesIO()
    seg.set_channels(1).set_frame_rate(16000).export(buf, format='wav')
    buf.seek(0)
    r = requests.post(
        URL,
        headers={'api-subscription-key': API_KEY},
        files={'file': ('chunk.wav', buf, 'audio/wav')},
        data={'model': MODEL, 'mode': mode, 'language_code': LANGUAGE},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()

step = CHUNK_SECONDS * 1000
native, english, langs = [], [], []
for i, start in enumerate(range(0, len(audio), step)):
    seg = audio[start:start + step]
    t = call_sarvam(seg, 'transcribe')   # original language
    e = call_sarvam(seg, 'translate')    # English
    native.append(t.get('transcript', '').strip())
    english.append(e.get('transcript', '').strip())
    langs.append(t.get('language_code'))
    print(f'chunk {i+1} done ({t.get("language_code")})')

native_text = ' '.join(p for p in native if p)
english_text = ' '.join(p for p in english if p)
detected = sorted({l for l in langs if l})

print('\nDetected language(s):', ', '.join(detected))
print('\n--- Original ---\n' + native_text)
print('\n--- English ---\n' + english_text)

In [ ]:
out = path.rsplit('.', 1)[0] + '.txt'
with open(out, 'w', encoding='utf-8') as f:
    f.write('Detected language(s): ' + ', '.join(detected) + '\n\n')
    f.write('--- Original ---\n' + native_text + '\n\n')
    f.write('--- English ---\n' + english_text + '\n')
files.download(out)